<a href="https://colab.research.google.com/github/brando710/brando710.github.io/blob/main/HW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
// ============================================================
//  ESP32-CAM HIGH-PERFORMANCE MJPEG STREAM (DUAL CORE)
//  Optimized for MAX FPS & LOW LATENCY
// ============================================================

#include "esp_camera.h"
#include <WiFi.h>
#include "esp_wifi.h"
#include "esp_bt.h"

// ============================================================
// WiFi
// ============================================================
const char* ssid     = "NETGEAR95";
const char* password = "grandbanana337";

WiFiServer server(80);

// ============================================================
// Camera Pins (AI Thinker)
// ============================================================
#define PWDN_GPIO_NUM   32
#define RESET_GPIO_NUM  -1
#define XCLK_GPIO_NUM    0
#define SIOD_GPIO_NUM   26
#define SIOC_GPIO_NUM   27
#define Y9_GPIO_NUM     35
#define Y8_GPIO_NUM     34
#define Y7_GPIO_NUM     39
#define Y6_GPIO_NUM     36
#define Y5_GPIO_NUM     21
#define Y4_GPIO_NUM     19
#define Y3_GPIO_NUM     18
#define Y2_GPIO_NUM      5
#define VSYNC_GPIO_NUM  25
#define HREF_GPIO_NUM   23
#define PCLK_GPIO_NUM   22

// ============================================================
// Frame Queue
// ============================================================
#include "freertos/queue.h"

typedef struct {
    uint8_t *buf;
    size_t len;
} frame_t;

QueueHandle_t frameQueue;

// ============================================================
// Camera Setup (Speed Optimized)
// ============================================================
void setupCamera() {
    camera_config_t cfg;

    cfg.ledc_channel = LEDC_CHANNEL_0;
    cfg.ledc_timer   = LEDC_TIMER_0;

    cfg.pin_d0       = Y2_GPIO_NUM;
    cfg.pin_d1       = Y3_GPIO_NUM;
    cfg.pin_d2       = Y4_GPIO_NUM;
    cfg.pin_d3       = Y5_GPIO_NUM;
    cfg.pin_d4       = Y6_GPIO_NUM;
    cfg.pin_d5       = Y7_GPIO_NUM;
    cfg.pin_d6       = Y8_GPIO_NUM;
    cfg.pin_d7       = Y9_GPIO_NUM;
    cfg.pin_xclk     = XCLK_GPIO_NUM;
    cfg.pin_pclk     = PCLK_GPIO_NUM;
    cfg.pin_vsync    = VSYNC_GPIO_NUM;
    cfg.pin_href     = HREF_GPIO_NUM;
    cfg.pin_sccb_sda = SIOD_GPIO_NUM;
    cfg.pin_sccb_scl = SIOC_GPIO_NUM;
    cfg.pin_pwdn     = PWDN_GPIO_NUM;
    cfg.pin_reset    = RESET_GPIO_NUM;

    cfg.xclk_freq_hz = 20000000;
    cfg.pixel_format = PIXFORMAT_JPEG;

    // 🔥 SPEED SETTINGS
    cfg.frame_size   = FRAMESIZE_QQVGA;   // fastest
    cfg.jpeg_quality = 20;                // lower = faster
    cfg.fb_count     = 2;
    cfg.grab_mode    = CAMERA_GRAB_LATEST;
    cfg.fb_location  = CAMERA_FB_IN_PSRAM;

    if (esp_camera_init(&cfg) != ESP_OK) {
        Serial.println("❌ Camera init failed");
        return;
    }

    sensor_t *s = esp_camera_sensor_get();
    if (s) {
        // Disable slow auto features
        s->set_gain_ctrl(s, 0);
        s->set_exposure_ctrl(s, 0);
        s->set_awb_gain(s, 0);

        s->set_vflip(s, 1);
        s->set_hmirror(s, 1);
    }

    Serial.println("✅ Camera ready (FAST MODE)");
}

// ============================================================
// Camera Task (Core 0)
// ============================================================
void cameraTask(void *pvParameters) {
    while (true) {
        camera_fb_t *fb = esp_camera_fb_get();
        if (!fb) continue;

        frame_t frame;
        frame.buf = (uint8_t*)malloc(fb->len);

        if (frame.buf) {
            memcpy(frame.buf, fb->buf, fb->len);
            frame.len = fb->len;

            // Drop frame if queue full (low latency)
            if (xQueueSend(frameQueue, &frame, 0) != pdTRUE) {
                free(frame.buf);
            }
        }

        esp_camera_fb_return(fb);
    }
}

// ============================================================
// Stream Task (Core 1)
// ============================================================
void streamTask(void *pvParameters) {
    WiFiClient client;

    while (true) {
        client = server.available();

        if (!client) {
            vTaskDelay(10);
            continue;
        }

        Serial.println("📡 Client connected");

        client.setNoDelay(true);

        client.print(
            "HTTP/1.1 200 OK\r\n"
            "Content-Type: multipart/x-mixed-replace; boundary=frame\r\n"
            "Cache-Control: no-cache\r\n\r\n"
        );

        frame_t frame;

        while (client.connected()) {
            if (xQueueReceive(frameQueue, &frame, portMAX_DELAY)) {
                client.printf(
                    "--frame\r\nContent-Type: image/jpeg\r\nContent-Length: %u\r\n\r\n",
                    frame.len
                );

                client.write(frame.buf, frame.len);
                client.print("\r\n");

                free(frame.buf);
            }
        }

        client.stop();
        Serial.println("❌ Client disconnected");
    }
}

// ============================================================
// Setup
// ============================================================
void setup() {
    Serial.begin(115200);
    delay(100);

    Serial.println("\n🚀 ESP32-CAM HIGH SPEED STREAM");

    // Max performance tweaks
    setCpuFrequencyMhz(240);
    esp_bt_controller_deinit();

    setupCamera();

    // WiFi tuning
    WiFi.mode(WIFI_STA);
    esp_wifi_set_ps(WIFI_PS_NONE);
    esp_wifi_set_max_tx_power(78);

    WiFi.begin(ssid, password);

    Serial.print("🔌 Connecting");
    while (WiFi.status() != WL_CONNECTED) {
        delay(500);
        Serial.print(".");
    }

    Serial.println(" ✅");
    Serial.print("📡 IP: ");
    Serial.println(WiFi.localIP());

    server.begin();

    // Create queue (small = low latency)
    frameQueue = xQueueCreate(2, sizeof(frame_t));

    // Start tasks on separate cores
    xTaskCreatePinnedToCore(cameraTask, "Camera Task", 4096, NULL, 2, NULL, 0);
    xTaskCreatePinnedToCore(streamTask, "Stream Task", 4096, NULL, 1, NULL, 1);

    Serial.println("🎥 Stream ready");
    Serial.printf("👉 http://%s\n", WiFi.localIP().toString().c_str());
}

// ============================================================
// Loop (unused)
// ============================================================
void loop() {
    delay(1000);
}

Improved code